# Run if libraries are not installed


In [ ]:
# %pip uninstall -y numpy numpy-base pandas

# # Hard-remove any leftovers that %pip might not clean
# import shutil, sys, glob, os
# for p in glob.glob("/usr/local/lib/python3.11/dist-packages/numpy*"):
#     shutil.rmtree(p, ignore_errors=True)

# # Optional: clear pip cache to avoid weird cache reuse
# !rm -rf /root/.cache/pip
# !pip install --force-reinstall --no-cache-dir numpy==2.0.1 pandas==2.2.2 scikit-learn==1.5.1 xgboost==2.1.1 scipy==1.13.1

# 1.Libraries

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd
import gc
import pickle
from tqdm.auto import tqdm
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from xgboost.callback import EarlyStopping
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pickle


sys.path.append("/kaggle/input/test-hack")
from metrics import evaluate_all_metrics
sys.path.append("/kaggle/input/test-hack")
from metrics import evaluate_all_metrics


print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Has local numpy shadowing?", os.path.exists("./numpy") or os.path.exists("./numpy.py"))
print("xgboost:", xgb.__version__)

NumPy: 2.0.2
Pandas: 2.2.2
Has local numpy shadowing? False
xgboost: 3.1.1


# 2.Files and Dataloaders

In [ ]:
# Insert File paths from dir over here
json_path = "/content/dataset_info.json"
train_path = "/content/train.pkl"
x_test_path  = "/content/x_test.pkl"
y_test_local_path = "/content/y_test_local.pkl"

def Extract_data(json_path: str, train_path: str, x_test_path: str, y_test_local_path: str):
    ROOT = Path.cwd().parent if (Path.cwd().name == 'src') else Path.cwd()
    DATA = ROOT
    SRC  = ROOT
    SUBM = ROOT

    # Ensure src is importable
    if str(SRC) not in sys.path:
        sys.path.insert(0, str(SRC))

    # Create sample_submission dir if missing
    SUBM.mkdir(parents=True, exist_ok=True)

    SEED = 1337
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Load JSON
    with open(json_path, "r") as f:
        data_info = json.load(f)
    print("Top-level keys:", list(data_info.keys()))
    print("\nPreview:")
    print(json.dumps(data_info, indent=2)[:1000])  # Show first 1000 chars
    # Peek train / x_test
    # train_path = DATA / "train.pkl"
    # x_test_path  = DATA / "x_test.pkl"
    # y_local_path = DATA / "y_test_local.pkl"

    train = pd.read_pickle(train_path)
    x_test  = pd.read_pickle(x_test_path)
    y_test_local = pd.read_pickle(y_test_local_path)

    print("train shape:", train.shape, "| columns:", train.columns.tolist())
    print("x_test  shape:", x_test.shape,  "| columns:", x_test.columns.tolist())
    print("y_test_local shape:", y_test_local.shape, "| columns:", y_test_local.columns.tolist())

    display(train.head(3))
    display(x_test.head(3))
    display(y_test_local.head(3))
    return train,x_test,y_test_local
train,x_test,y_test_local=Extract_data(json_path,train_path,x_test_path,y_test_local_path)

Top-level keys: ['freq', 'features', 'input_len', 'horizon_len', 'dtypes', 'outputs', 'sha256']

Preview:
{
  "freq": "1min",
  "features": [
    "close",
    "volume"
  ],
  "input_len": 60,
  "horizon_len": 10,
  "dtypes": {
    "close": "float32",
    "volume": "float32"
  },
  "outputs": {
    "train": {
      "columns": [
        "series_id",
        "time_step",
        "close",
        "volume"
      ]
    },
    "x_test": {
      "columns": [
        "window_id",
        "time_step",
        "close",
        "volume"
      ]
    },
    "y_test_local": {
      "columns": [
        "window_id",
        "time_step",
        "close"
      ]
    }
  },
  "sha256": {
    "train.pkl": "f18af2ae073ccf53747d57d9607304dd04fd7dc3e04d6d4a0143776a44044527",
    "x_test.pkl": "220465762feb375b96ae5af41a9e910f39a2bf7bed3a176b43d21cdfcda5aec5",
    "y_test_local.pkl": "bd40d6806e27ce47d7bb35e2e3638bf02c2b868c6d78a70240aef84a82b6bfcd"
  }
}
train shape: (18331224, 4) | columns: ['series_id', 't

,series_id,time_step,close,volume
0,1,0,0.13700,171985.703125
1,1,1,0.13656,85451.398438
2,1,2,0.13647,121151.898438


,window_id,time_step,close,volume
0,1,0,0.1126,24976.0
1,1,1,0.1126,0.0
2,1,2,0.1125,2299.0


,window_id,time_step,close
0,1,0,0.1131
1,1,1,0.1131
2,1,2,0.1130


# TOKEN Descriptive Statistics

In [ ]:
#train['close']=np.log(train['close'])
train.groupby(['series_id']).agg({
     'close': ['min', 'max','mean'],
}).sort_values(('close', 'mean'), ascending=False)

close                            
                    min           max          mean
series_id                                          
32         16505.869141  31798.000000  26320.408203
4           3810.780029  12450.219727   9173.823242
33            58.000000    623.869995    290.729248
23            33.189999    161.389999     81.160538
42            13.870000     61.009998     31.903854
7              5.007800     41.269001     16.192724
3              7.380000     41.700001     15.354797
31             2.893000     20.440001      8.523156
28             1.129900     21.243000      7.860170
6              2.459000      8.994000      5.217080
45             1.291100     11.835000      4.189503
15             0.874000     22.497999      2.607200
36             0.954000      4.974000      2.581236
17             0.533000      4.639000      2.258248
24             0.914000      3.758200      2.053357
19             0.538000      6.750000      1.988276
26             1.222000      3.378000      1.936351
25             1.029000      2.844000      1.694237
43             0.780000      2.023000      1.380660
13             0.379000      1.576000      0.792221
10             0.388500      1.298000      0.790860
5              0.404000      1.633000      0.746253
22             0.326500      0.968900      0.579561
2              0.260900      0.939100      0.533405
29             0.316700      0.909100      0.483857
50             0.276500      0.808400      0.476090
39             0.180400      0.878700      0.438050
34             0.276900      0.666000      0.434114
37             0.124120      0.795990      0.373588
46             0.144200      1.032900      0.370632
8              0.158600      0.572300      0.348513
49             0.105890      0.345200      0.222374
12             0.004607      0.738050      0.197726
38             0.041790      0.597750      0.170354
14             0.101180      0.168400      0.127296
41             0.075800      0.162300      0.109146
48             0.049330      0.207200      0.103289
1              0.048440      0.184960      0.100583
9              0.039840      0.147890      0.081676
27             0.026380      0.179970      0.074909
47             0.055100      0.103480      0.074141
44             0.017970      0.154430      0.073076
18             0.049820      0.093730      0.071260
40             0.026170      0.119410      0.070465
11             0.036150      0.167930      0.052794
21             0.034030      0.085650      0.052439
30             0.014680      0.036910      0.023588
35             0.002250      0.029890      0.012073
16             0.000580      0.031600      0.007929
20             0.000985      0.002473      0.001593

# 3.Functions for overlapping vs non-overlapping windows

In [ ]:
def make_overlapping_windows(train: pd.DataFrame, window: int = 70) -> pd.DataFrame:
    """
    Build overlapping (stride=1) windows of length `window` per series_id.
    Output columns:
      series_id,
      timestamp_1_close ... timestamp_{window}_close,
      timestamp_1_vol   ... timestamp_{window}_vol
    """
    # Basic checks
    required = {"series_id", "time_step", "close", "volume"}
    missing = required - set(train.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    # Sort to ensure correct temporal ordering
    train = train.sort_values(["series_id", "time_step"], kind="mergesort").reset_index(drop=True)

    # Pre-build column names
    close_cols = [f"timestamp_{i}_close" for i in range(1, window + 1)]
    vol_cols   = [f"timestamp_{i}_vol"   for i in range(1, window + 1)]
    all_cols   = ["series_id"] + close_cols + vol_cols

    out_frames = []
    for sid, g in train.groupby("series_id", sort=False):
        n = len(g)
        if n < window:
            continue  # not enough points for a full window

        # Create overlapping windows using a zero-copy view
        c_win = np.lib.stride_tricks.sliding_window_view(g["close"].to_numpy(),  window_shape=window)
        v_win = np.lib.stride_tricks.sliding_window_view(g["volume"].to_numpy(), window_shape=window)

        # Concatenate close and volume along the feature axis -> (num_windows, 2*window)
        X = np.concatenate([c_win, v_win], axis=1)

        # Build a DataFrame for this series
        dfw = pd.DataFrame(X, columns=close_cols + vol_cols)
        dfw.insert(0, "series_id", sid)
        out_frames.append(dfw)

    if not out_frames:
        # Return empty with correct schema if nothing qualified
        return pd.DataFrame(columns=all_cols)

    return pd.concat(out_frames, ignore_index=True)

def make_non_overlapping_windows(train: pd.DataFrame, window: int = 70, drop_incomplete: bool = True) -> pd.DataFrame:
    """
    Build non-overlapping windows of length `window` per series_id.
    Output columns:
      series_id,
      timestamp_1_close ... timestamp_{window}_close,
      timestamp_1_vol   ... timestamp_{window}_vol

    If `drop_incomplete` is True, discard any trailing partial window.
    If False, the final partial window is padded with NaN to length `window`.
    """
    # Basic checks
    required = {"series_id", "time_step", "close", "volume"}
    missing = required - set(train.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    # Sort to ensure correct temporal ordering
    train = train.sort_values(["series_id", "time_step"], kind="mergesort").reset_index(drop=True)

    # Pre-build column names
    close_cols = [f"timestamp_{i}_close" for i in range(1, window + 1)]
    vol_cols   = [f"timestamp_{i}_vol"   for i in range(1, window + 1)]
    all_cols   = ["series_id"] + close_cols + vol_cols

    out_frames = []

    for sid, g in train.groupby("series_id", sort=False):
        n = len(g)
        if n < window and drop_incomplete:
            continue  # not enough points for even one full window

        c = g["close"].to_numpy()
        v = g["volume"].to_numpy()

        if drop_incomplete:
            n_full = (n // window) * window
            if n_full == 0:
                continue
            c = c[:n_full].reshape(-1, window)
            v = v[:n_full].reshape(-1, window)
        else:
            # Pad the last partial window with NaN (if any)
            rem = n % window
            if rem != 0:
                pad = window - rem
                c = np.pad(c, (0, pad), constant_values=np.nan)
                v = np.pad(v, (0, pad), constant_values=np.nan)
            c = c.reshape(-1, window)
            v = v.reshape(-1, window)

        # Concatenate close and volume along the feature axis -> (num_windows, 2*window)
        X = np.concatenate([c, v], axis=1)

        # Build a DataFrame for this series
        dfw = pd.DataFrame(X, columns=close_cols + vol_cols)
        dfw.insert(0, "series_id", sid)
        out_frames.append(dfw)

    if not out_frames:
        # Return empty with correct schema if nothing qualified
        return pd.DataFrame(columns=all_cols)

    return pd.concat(out_frames, ignore_index=True).dropna()


# 4.Utility Functions

In [ ]:
def Extract_Predictions(models,df):
    y_pred = {}
    if "window_id" in df.columns:
        df_dropped = df.drop(columns=["window_id"])
        for tgt, mdl in models.items():
            y_pred[tgt] = mdl.predict(df_dropped)
        y_pred_df = pd.DataFrame(y_pred, index=getattr(df, "index", None))
        y_pred_df['window_id']=df['window_id']
        return y_pred_df
    else:
        for tgt, mdl in models.items():
            y_pred[tgt] = mdl.predict(df)
        y_pred_df = pd.DataFrame(y_pred, index=getattr(df, "index", None))
        y_pred_df["window_id"] = range(1, len(y_pred_df) + 1)
        return y_pred_df

def Preprocess_before_predictions(df):
    pivoted = (
        df
        .pivot(index="window_id", columns="time_step", values=["close", "volume"])
        .sort_index(axis=1, level=1)
    )
    pivoted = pivoted.reindex(columns=["close", "volume"], level=0)
    # Flatten columns — rename 'volume' → 'vol'
    pivoted.columns = [
        f"timestamp_{t+1}_{'vol' if var == 'volume' else var}"
        for var, t in pivoted.columns
    ]
    #Call feature engineering function
    return feature_engineering(pivoted.reset_index())

def Convert_in_submission_df(df):
    y_long = df.melt(
        id_vars='window_id',
        var_name='time_step',
        value_name='pred_close'
    )
    # # # Extract numeric part from 'target_x' columns
    y_long['time_step'] = y_long['time_step'].str.extract('(\d+)').astype(int) - 1
    # # # Rename and reorder columns
    # y_long = y_long.rename(columns={'index': 'window_id'})
    # y_long['window_id'] += 1  # to start IDs from 1 instead of 0
    y_long = y_long.sort_values(['window_id', 'time_step']).reset_index(drop=True)
    return y_long

<>:39: SyntaxWarning: invalid escape sequence '\d'
<>:39: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipython-input-2781487806.py:39: SyntaxWarning: invalid escape sequence '\d'
  y_long['time_step'] = y_long['time_step'].str.extract('(\d+)').astype(int) - 1


# 5.Preprocessing and Target Creation

In [ ]:
df=make_non_overlapping_windows(train,70,False)
#df['mean_close'] = df.loc[:, 'timestamp_1_close':'timestamp_60_close'].mean(axis=1)
# df = df[df['mean_close'] >= np.log(1)]
#df = df[df['mean_close'] <= np.log(1)]
#df = df.drop(columns=['mean_close'])


# Create target metrics
cleaned_df = df.drop(["timestamp_61_vol","timestamp_62_vol","timestamp_63_vol","timestamp_64_vol","timestamp_65_vol","timestamp_66_vol","timestamp_67_vol","timestamp_68_vol","timestamp_69_vol","timestamp_70_vol"], axis=1)



old_cols = [
    "timestamp_61_close","timestamp_62_close","timestamp_63_close",
    "timestamp_64_close","timestamp_65_close","timestamp_66_close",
    "timestamp_67_close","timestamp_68_close","timestamp_69_close",
    "timestamp_70_close"
]
new_cols = {old: f"target_{i+1}" for i, old in enumerate(old_cols)}
cleaned_df = cleaned_df.rename(columns=new_cols)
#Remove DF to create space
del df
gc.collect()
cleaned_df.head()
target_list=["target_1","target_2","target_3","target_4","target_5","target_6","target_7","target_8","target_9","target_10"]
# cleaned_df = cleaned_df.sample(frac=0.7, random_state=42).reset_index(drop=True)

In [ ]:
cleaned_df.groupby('series_id').size().reset_index(name='count')

,series_id,count
0,1,5636
1,2,5636
2,3,5616
3,4,5623
4,5,5616
5,6,5636
6,7,5601
7,8,5614
8,9,4898
9,10,5614


# 6.Train test split.
We are taking latest windows for each token for training. Below code includes and excludes certain tokens based on data available

In [ ]:
step=len(cleaned_df)
if round((step*0.3)/2)%2!=0:
    step=round((step*0.3)/2)
    step=step+1
else:
    step=round((step*0.3)/2)
step=round(step/len(cleaned_df['series_id'].unique()))
print(step)
series_counts = cleaned_df.groupby('series_id').size().reset_index(name='count')

# Print series_ids with count less than step
print(series_counts[series_counts['count'] < step])

# Create a list of those series_ids
exclude_ids = series_counts.loc[series_counts['count'] < step, 'series_id'].tolist()

print("exclude_ids =", exclude_ids)

786
    series_id  count
21         22    549
exclude_ids = [22]


In [ ]:
train_df = (
    cleaned_df
    .groupby('series_id', group_keys=False)
    .apply(lambda x: x.iloc[:-step])
    .reset_index(drop=True)
)

test_df = (
    cleaned_df
    .groupby('series_id', group_keys=False)
    .apply(lambda x: x.tail(step))
    .reset_index(drop=True)
)

del cleaned_df
gc.collect()

# Exclude the unwanted series_ids from test, validation, and holdout
test_df = test_df[~test_df['series_id'].isin(exclude_ids)]

validation_set = (
    test_df
    .groupby('series_id', group_keys=False)
    .apply(lambda x: x.head(step // 2))
    .reset_index(drop=True)
)

holdout_set = (
    test_df
    .groupby('series_id', group_keys=False)
    .apply(lambda x: x.tail(step // 2))
    .reset_index(drop=True)
)

/tmp/ipython-input-27502882.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.iloc[:-step])
/tmp/ipython-input-27502882.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.tail(step))
/tmp/ipython-input-27502882.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the groupi

In [ ]:
holdout_set.to_csv("my_data.csv", index=False)

# 7.Custom min max scalor
scales data at token level

In [ ]:
def row_minmax_scaler(df, save_path=None):
    """
    Row-wise MinMax scaling with two criteria sets:
    1. Based on timestamp_*_close for close + target columns
    2. Based on timestamp_*_vol for vol columns
    """

    # Define column groups
    close_cols = [f'timestamp_{i}_close' for i in range(1, 61)]
    vol_cols   = [f'timestamp_{i}_vol' for i in range(1, 61)]
    target_cols = [f'target_{i}' for i in range(1, 11)]

    # --- Compute row-wise min/max for close and vol ---
    row_min_close = df.loc[:, close_cols].min(axis=1)
    row_max_close = df.loc[:, close_cols].max(axis=1)

    row_min_vol = df.loc[:, vol_cols].min(axis=1)
    row_max_vol = df.loc[:, vol_cols].max(axis=1)

    # --- Store scaling criteria ---
    criteria = pd.DataFrame({
        'row_min_close': row_min_close,
        'row_max_close': row_max_close,
        'row_min_vol': row_min_vol,
        'row_max_vol': row_max_vol
    }, index=df.index)

    # --- Scale ---
    df_scaled = df.copy()

    # Scale close + target columns using close min/max
    denom_close = (criteria['row_max_close'] - criteria['row_min_close']).replace(0, np.nan)
    df_scaled[close_cols + target_cols] = (
        df[close_cols + target_cols].sub(criteria['row_min_close'], axis=0)
        .div(denom_close, axis=0)
    )

    # Scale vol columns using vol min/max
    denom_vol = (criteria['row_max_vol'] - criteria['row_min_vol']).replace(0, np.nan)
    df_scaled[vol_cols] = (
        df[vol_cols].sub(criteria['row_min_vol'], axis=0)
        .div(denom_vol, axis=0)
    )

    # --- Optional save ---
    if save_path:
        criteria.to_csv(save_path, index=False)
        print(f"✅ Saved scaling criteria to {save_path}")

    return df_scaled, criteria

## Stores criteria for later use

In [ ]:
train_df,train_df_cr=row_minmax_scaler(train_df, save_path="train_df.csv")
validation_set,validation_set_cr=row_minmax_scaler(validation_set, save_path="validation_set.csv")
holdout_set,holdout_set_cr=row_minmax_scaler(holdout_set, save_path="holdout_set.csv")


✅ Saved scaling criteria to train_df.csv
✅ Saved scaling criteria to validation_set.csv
✅ Saved scaling criteria to holdout_set.csv


In [ ]:
print(len(train_df))
print(len(validation_set))
print(len(holdout_set))
print(len(train_df.dropna()))
print(len(validation_set.dropna()))
print(len(holdout_set.dropna()))
#train_df.dropna()
train_df=train_df.dropna()
validation_set=validation_set.dropna()
holdout_set=holdout_set.dropna()

222781
19257
19257
222776
19257
19257


In [ ]:
print(len(holdout_set)+len(validation_set))
print(len(train_df))


38514
222776


# 8.Feature engineering function.

In [ ]:
def feature_engineering(df):
    #### Rate of change ####
    roc_Feature_list=[]
    var_list=['vol','close']
    time_frame=['2','5','10','15','30','60']
    for var in var_list:
        for time in time_frame:
            roc_Feature_list.append('rate_of_change_'+var+'_L'+time+'M')
            df[f'rate_of_change_{var}_L{time}M']= np.where(df[f'timestamp_{time}_{var}'] == 0, 0,(df[f'timestamp_1_{var}'] - df[f'timestamp_{time}_{var}']) / df[f'timestamp_{time}_{var}'])
    print("ROC Features:",roc_Feature_list)

    #### Statistical Features ####
    stat_Feature_list = []
    var_list = ['vol', 'close']
    for var in var_list:
        for t in range(5,65,5): #range(5,65,5)
                # Find the last `t` columns that match the variable name pattern
            cols = ["timestamp_"+str(60-c)+"_"+var for c in range(t)]
            df[f'rolling_mean_{var}_{t}'] = df[cols].mean(axis=1)
            stat_Feature_list.append(f'rolling_mean_{var}_{t}')
            df[f'rolling_var_{var}_{t}'] = df[cols].std(axis=1)
            stat_Feature_list.append(f'rolling_var_{var}_{t}')
    print("Statistical Features:",stat_Feature_list)
    # ### VMAP ####
    # VMAP_Feature_list=[]
    # var_list = ['vol', 'close']
    # for t in range(5, 65, 5):
    #     # Select last `t` timestamps for both variables
    #     close_cols = [f"timestamp_{60-c}_close" for c in range(t)]
    #     vol_cols   = [f"timestamp_{60-c}_vol"   for c in range(t)]

    #     # --- VWAP computation ---
    #     # VWAP = sum(price * volume) / sum(volume)
    #     df[f'vwap_{t}'] = (
    #         (df[close_cols].values * df[vol_cols].values).sum(axis=1) /
    #         df[vol_cols].sum(axis=1)
    #     )
    #     VMAP_Feature_list.append(f'vwap_{t}')
    # print("VWAP Features:",VMAP_Feature_list)

    # ### WMAP ####
    # WMAP_Feature_list = []
    # var_list = ['vol', 'close']

    # for t in range(5, 65, 5):
    #     # Select last `t` timestamps for both variables
    #     close_cols = [f"timestamp_{60-c}_close" for c in range(t)]
    #     vol_cols   = [f"timestamp_{60-c}_vol"   for c in range(t)]

    #     # Create weights (1 to t) for recency emphasis
    #     weights = np.arange(1, t + 1)

    #     # --- WMAP computation ---
    #     # WMAP = sum(w * price * volume) / sum(w * volume)
    #     num = (df[close_cols].values * df[vol_cols].values * weights).sum(axis=1)
    #     den = (df[vol_cols].values * weights).sum(axis=1)
    #     df[f'wmap_{t}'] = num / den

    #     WMAP_Feature_list.append(f'wmap_{t}')

    # print("WMAP Features:", WMAP_Feature_list)
    # #### Slope Features ####
    slope_Feature_list=[]
    for slope in [i for i in range(15,65,5)]:#[15,30,45,60]:
        cols_vol=["timestamp_"+str(60-c)+"_"+"vol" for c in range(slope)]
        cols_close=["timestamp_"+str(60-c)+"_"+"close" for c in range(slope)]
        #(df[cols] - df[cols].mean()).prod(axis=1)
        X = df[cols_vol].to_numpy()
        Y = df[cols_close].to_numpy()
        # Subtract mean (centered regression)
        X_centered = X - X.mean(axis=1, keepdims=True)
        Y_centered = Y - Y.mean(axis=1, keepdims=True)
        # Compute slope per row:  Σ(x*y) / Σ(x²)
        slopes = (X_centered * Y_centered).sum(axis=1) / (X_centered**2).sum(axis=1)
        df[f'slope_{slope}'] = slopes
        slope_Feature_list.append('slope_'+str(slope))
    print("Slope Features:",slope_Feature_list)
    return df
train_df=feature_engineering(train_df)
validation_set=feature_engineering(validation_set)
holdout_set=feature_engineering(holdout_set)

ROC Features: ['rate_of_change_vol_L2M', 'rate_of_change_vol_L5M', 'rate_of_change_vol_L10M', 'rate_of_change_vol_L15M', 'rate_of_change_vol_L30M', 'rate_of_change_vol_L60M', 'rate_of_change_close_L2M', 'rate_of_change_close_L5M', 'rate_of_change_close_L10M', 'rate_of_change_close_L15M', 'rate_of_change_close_L30M', 'rate_of_change_close_L60M']


/tmp/ipython-input-865061860.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'rate_of_change_{var}_L{time}M']= np.where(df[f'timestamp_{time}_{var}'] == 0, 0,(df[f'timestamp_1_{var}'] - df[f'timestamp_{time}_{var}']) / df[f'timestamp_{time}_{var}'])
/tmp/ipython-input-865061860.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'rate_of_change_{var}_L{time}M']= np.where(df[f'timestamp_{time}_{var}'] == 0, 0,(df[f'timestamp_1_{var}'] - df[f'timestamp_{time}_{var}']) / df[f'timestamp_{time}_{var}'])
/tmp/ipython-input

Statistical Features: ['rolling_mean_vol_5', 'rolling_var_vol_5', 'rolling_mean_vol_10', 'rolling_var_vol_10', 'rolling_mean_vol_15', 'rolling_var_vol_15', 'rolling_mean_vol_20', 'rolling_var_vol_20', 'rolling_mean_vol_25', 'rolling_var_vol_25', 'rolling_mean_vol_30', 'rolling_var_vol_30', 'rolling_mean_vol_35', 'rolling_var_vol_35', 'rolling_mean_vol_40', 'rolling_var_vol_40', 'rolling_mean_vol_45', 'rolling_var_vol_45', 'rolling_mean_vol_50', 'rolling_var_vol_50', 'rolling_mean_vol_55', 'rolling_var_vol_55', 'rolling_mean_vol_60', 'rolling_var_vol_60', 'rolling_mean_close_5', 'rolling_var_close_5', 'rolling_mean_close_10', 'rolling_var_close_10', 'rolling_mean_close_15', 'rolling_var_close_15', 'rolling_mean_close_20', 'rolling_var_close_20', 'rolling_mean_close_25', 'rolling_var_close_25', 'rolling_mean_close_30', 'rolling_var_close_30', 'rolling_mean_close_35', 'rolling_var_close_35', 'rolling_mean_close_40', 'rolling_var_close_40', 'rolling_mean_close_45', 'rolling_var_close_45', 

/tmp/ipython-input-865061860.py:74: RuntimeWarning: invalid value encountered in divide
  slopes = (X_centered * Y_centered).sum(axis=1) / (X_centered**2).sum(axis=1)
/tmp/ipython-input-865061860.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'slope_{slope}'] = slopes
/tmp/ipython-input-865061860.py:74: RuntimeWarning: invalid value encountered in divide
  slopes = (X_centered * Y_centered).sum(axis=1) / (X_centered**2).sum(axis=1)
/tmp/ipython-input-865061860.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f's

Slope Features: ['slope_15', 'slope_20', 'slope_25', 'slope_30', 'slope_35', 'slope_40', 'slope_45', 'slope_50', 'slope_55', 'slope_60']
ROC Features: ['rate_of_change_vol_L2M', 'rate_of_change_vol_L5M', 'rate_of_change_vol_L10M', 'rate_of_change_vol_L15M', 'rate_of_change_vol_L30M', 'rate_of_change_vol_L60M', 'rate_of_change_close_L2M', 'rate_of_change_close_L5M', 'rate_of_change_close_L10M', 'rate_of_change_close_L15M', 'rate_of_change_close_L30M', 'rate_of_change_close_L60M']
Statistical Features: ['rolling_mean_vol_5', 'rolling_var_vol_5', 'rolling_mean_vol_10', 'rolling_var_vol_10', 'rolling_mean_vol_15', 'rolling_var_vol_15', 'rolling_mean_vol_20', 'rolling_var_vol_20', 'rolling_mean_vol_25', 'rolling_var_vol_25', 'rolling_mean_vol_30', 'rolling_var_vol_30', 'rolling_mean_vol_35', 'rolling_var_vol_35', 'rolling_mean_vol_40', 'rolling_var_vol_40', 'rolling_mean_vol_45', 'rolling_var_vol_45', 'rolling_mean_vol_50', 'rolling_var_vol_50', 'rolling_mean_vol_55', 'rolling_var_vol_55', 

/tmp/ipython-input-865061860.py:74: RuntimeWarning: invalid value encountered in divide
  slopes = (X_centered * Y_centered).sum(axis=1) / (X_centered**2).sum(axis=1)
/tmp/ipython-input-865061860.py:74: RuntimeWarning: invalid value encountered in divide
  slopes = (X_centered * Y_centered).sum(axis=1) / (X_centered**2).sum(axis=1)
/tmp/ipython-input-865061860.py:74: RuntimeWarning: invalid value encountered in divide
  slopes = (X_centered * Y_centered).sum(axis=1) / (X_centered**2).sum(axis=1)
/tmp/ipython-input-865061860.py:74: RuntimeWarning: invalid value encountered in divide
  slopes = (X_centered * Y_centered).sum(axis=1) / (X_centered**2).sum(axis=1)
/tmp/ipython-input-865061860.py:74: RuntimeWarning: invalid value encountered in divide
  slopes = (X_centered * Y_centered).sum(axis=1) / (X_centered**2).sum(axis=1)


Statistical Features: ['rolling_mean_vol_5', 'rolling_var_vol_5', 'rolling_mean_vol_10', 'rolling_var_vol_10', 'rolling_mean_vol_15', 'rolling_var_vol_15', 'rolling_mean_vol_20', 'rolling_var_vol_20', 'rolling_mean_vol_25', 'rolling_var_vol_25', 'rolling_mean_vol_30', 'rolling_var_vol_30', 'rolling_mean_vol_35', 'rolling_var_vol_35', 'rolling_mean_vol_40', 'rolling_var_vol_40', 'rolling_mean_vol_45', 'rolling_var_vol_45', 'rolling_mean_vol_50', 'rolling_var_vol_50', 'rolling_mean_vol_55', 'rolling_var_vol_55', 'rolling_mean_vol_60', 'rolling_var_vol_60', 'rolling_mean_close_5', 'rolling_var_close_5', 'rolling_mean_close_10', 'rolling_var_close_10', 'rolling_mean_close_15', 'rolling_var_close_15', 'rolling_mean_close_20', 'rolling_var_close_20', 'rolling_mean_close_25', 'rolling_var_close_25', 'rolling_mean_close_30', 'rolling_var_close_30', 'rolling_mean_close_35', 'rolling_var_close_35', 'rolling_mean_close_40', 'rolling_var_close_40', 'rolling_mean_close_45', 'rolling_var_close_45', 

/tmp/ipython-input-865061860.py:74: RuntimeWarning: invalid value encountered in divide
  slopes = (X_centered * Y_centered).sum(axis=1) / (X_centered**2).sum(axis=1)


# 9.Final train test and hold out creation

In [ ]:
train_df = train_df.drop(['series_id'], axis=1)
X = train_df.drop(columns=target_list)
y = train_df[target_list]

validation_set = validation_set.drop(['series_id'], axis=1)
validation_set_X = validation_set.drop(columns=target_list)
validation_set_y = validation_set[target_list]

holdout_set = holdout_set.drop(['series_id'], axis=1)
holdout_set_X = holdout_set.drop(columns=target_list)
holdout_set_y = holdout_set[target_list]
models = {}

# 10.Model Training (Using XGBOOST)

In [ ]:
def mse_eval(y_true, y_pred):
    return "mse", float(np.mean((y_true - y_pred) ** 2)),


def mse_eval_skl(y_true, y_pred):
    # y_true and y_pred are 1D arrays
    return float(np.mean((y_true - y_pred) ** 2))

In [ ]:
Training_brute_force_active=False
models = {}
if(Training_brute_force_active):
    for tgt in target_list[:1]:
        model = xgb.XGBRegressor(
            n_estimators=115,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            tree_method="hist",
            n_jobs=-1,
            early_stopping_rounds=50,
            eval_metric="rmse"   # <-- set here for XGB 2.x
        )
        model.fit(
            X, y[tgt],
            eval_set=[(validation_set_X, validation_set_y[tgt])],
            #callbacks=[es],
            verbose=True
        )
        print(tgt," is done")
        models[tgt] = model
else:
    for tgt in target_list:
        model = xgb.XGBRegressor(
            n_estimators= 500,
            learning_rate=  0.05391735228604129,
            max_depth= 5,
            min_child_weight= 0.01243414935667999,
            subsample= 0.5257787196856536,
            colsample_bytree= 0.8970277194290459,
            reg_alpha= 0.003116611164785481,
            reg_lambda=  0.13772558090106876,
            gamma= 2.7571616920507793,
            # n_estimators=1600,
            # learning_rate=0.05,
            # max_depth=4,
            # subsample=0.8,
            # colsample_bytree=0.8,
            # random_state=42,
            # tree_method="hist",
            # n_jobs=-1,
            early_stopping_rounds=150,
            eval_metric="rmse"   # <-- set here for XGB 2.x
        )
        model.fit(
            X, y[tgt],
            eval_set=[(validation_set_X, validation_set_y[tgt])],
            #callbacks=[es],
            verbose=False
        )
        print(tgt," is done")
        models[tgt] = model
# 'n_estimators': 500, 'learning_rate': 0.05391735228604129, 'max_depth': 5, 'min_child_weight': 0.01243414935667999, 'subsample': 0.5257787196856536, 'colsample_bytree': 0.8970277194290459, 'reg_alpha': 0.003116611164785481,

# import lightgbm as lgb
# Training_brute_force_active = False
# models = {}

# # Shared params (close to your XGB setup)
# common_params = dict(
#     learning_rate=0.05,
#     max_depth=5,                # pairs fine with default num_leaves=31
#     feature_fraction=0.8,       # == colsample_bytree
#     bagging_fraction=0.8,       # == subsample
#     bagging_freq=1,             # enable bagging
#     #reg_alpha=0.0,              # L1 (can tune)
#     #reg_lambda=1.0,             # L2 (can tune)
#     random_state=42,
#     n_jobs=-1,
#     objective="regression"      # LightGBM default for regressor
# )

# # LightGBM early stopping via callbacks is the most robust across versions
# early_stop_cb = lgb.early_stopping(stopping_rounds=50, verbose=True)

# if Training_brute_force_active:
#     for tgt in target_list[:1]:
#         model = lgb.LGBMRegressor(
#             n_estimators=1000,
#             **common_params
#         )
#         model.fit(
#             X, y[tgt],
#             eval_set=[(validation_set_X, validation_set_y[tgt])],
#             eval_metric="rmse",
#             callbacks=[early_stop_cb]
#         )
#         print(tgt, "is done")
#         models[tgt] = model
# else:
#     for tgt in target_list:
#         model = lgb.LGBMRegressor(
#             n_estimators=5000,
#             **common_params
#         )
#         model.fit(
#             X, y[tgt],
#             eval_set=[(validation_set_X, validation_set_y[tgt])],
#             eval_metric="rmse",
#             callbacks=[early_stop_cb]
#         )
#         print(tgt, "is done")
#         models[tgt] = model


target_1  is done
target_2  is done
target_3  is done
target_4  is done
target_5  is done
target_6  is done
target_7  is done
target_8  is done
target_9  is done
target_10  is done


#11.Inference

## Save as pkl file

In [ ]:
with open("model_weights.pkl", "wb") as f:
    pickle.dump(models, f)

## Inverse function of minmax

In [ ]:

def inverse_row_minmax(df_scaled, criteria):
    """
    Restores original close/vol/target values using stored row-wise criteria.
    """
    df_restored = df_scaled.copy()

    close_cols = [f'timestamp_{i}_close' for i in range(1, 61)]
    vol_cols   = [f'timestamp_{i}_vol' for i in range(1, 61)]
    target_cols = [f'target_{i}' for i in range(1, 11)]

    # Inverse for close + target
    df_restored[target_cols] = (
        df_scaled[target_cols]
        .mul(criteria['row_max_close'] - criteria['row_min_close'], axis=0)
        .add(criteria['row_min_close'], axis=0)
    )
    return df_restored


## Extract predictions on holdout

In [ ]:
holdout_x=Extract_Predictions(models,holdout_set_X)
holdout_x=inverse_row_minmax(holdout_x, holdout_set_cr)
pred_holdout=Convert_in_submission_df(holdout_x)
pred_holdout
#pred_holdout['pred_close']=np.exp(pred_holdout['pred_close'])

,window_id,time_step,pred_close
0,1,0,0.054664
1,1,1,0.054663
2,1,2,0.054661
3,1,3,0.054661
4,1,4,0.054661
...,...,...,...
192565,19257,5,0.376950
192566,19257,6,0.376964
192567,19257,7,0.376908
192568,19257,8,0.376925


## preprocessing for results comparision

In [ ]:
holdout_set_y['window_id'] = holdout_set_y.index + 1
holdout_set_Y=inverse_row_minmax(holdout_set_y, holdout_set_cr)
holdout_set_Y=Convert_in_submission_df(holdout_set_Y)
holdout_set_Y['close']=(holdout_set_Y['pred_close'])
holdout_set_Y = holdout_set_Y.drop('pred_close', axis=1)

/tmp/ipython-input-1769832009.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  holdout_set_y['window_id'] = holdout_set_y.index + 1


In [ ]:
holdout_set_Y

,window_id,time_step,close
0,1,0,0.05464
1,1,1,0.05464
2,1,2,0.05464
3,1,3,0.05464
4,1,4,0.05466
...,...,...,...
192565,19257,5,0.37680
192566,19257,6,0.37720
192567,19257,7,0.37700
192568,19257,8,0.37730


In [ ]:

off_stats = evaluate_all_metrics(
        y_true=holdout_set_Y,
        y_pred=pred_holdout,
        x_test=pd.DataFrame({"window_id": holdout_set_X.index + 1,"time_step": 59,"close": holdout_set_X["timestamp_50_close"]}).reset_index(drop=True),
        y_true_with_base=holdout_set_Y.merge(pred_holdout,on=["window_id", "time_step"],how="inner",suffixes=("", "_pred")).rename(columns={"pred_close": "base_close"}),
        horizon_step=0,
    )
display(pd.DataFrame([off_stats]).T.rename(columns={0: "value"}))

,value
MSE,2.369537e+01
MAE,5.527721e-01
IC,9.950707e-02
IR,3.313677e-01
SharpeRatio,4.082661e+06
MDD,0.000000e+00
VaR,4.082661e-06
ES,4.082661e-06


In [ ]:


# holdout_x=Extract_Predictions(models,holdout_set_X)
# pred_holdout=Convert_in_submission_df(holdout_x)
# pred_holdout['pred_close']=np.exp(pred_holdout['pred_close'])
# holdout_set_y['window_id'] = holdout_set_y.index + 1
# holdout_act_y=Convert_in_submission_df(holdout_set_y)
# #holdout_act_y['close']=np.exp(holdout_act_y['pred_close'])
# holdout_act_y = holdout_act_y.drop('pred_close', axis=1)
# #holdout_set_y = holdout_set_y.rename(columns={'pred_close': 'close'})
# sys.path.append("/kaggle/input/test-hack")
# from metrics import evaluate_all_metrics
# sys.path.append("/kaggle/input/test-hack")
# from metrics import evaluate_all_metrics
# off_stats = evaluate_all_metrics(
#         y_true=holdout_act_y,
#         y_pred=pred_holdout,
#         x_test=pd.DataFrame({"window_id": holdout_set_X.index + 1,"time_step": 59,"close": holdout_set_X["timestamp_50_close"]}).reset_index(drop=True),
#         y_true_with_base=holdout_act_y.merge(pred_holdout,on=["window_id", "time_step"],how="inner",suffixes=("", "_pred")).rename(columns={"pred_close": "base_close"}),
#         horizon_step=0,
#     )
# display(pd.DataFrame([off_stats]).T.rename(columns={0: "value"}))
# import pickle
# models_old = {}
# # Load the saved models
# with open("/kaggle/input/lightgbm-log/other/default/1/model_weights.pkl", "rb") as f:
#     models_old = pickle.load(f)
# holdout_x_old=Extract_Predictions(models_old,holdout_set_X)
# pred_holdout_old=Convert_in_submission_df(holdout_x_old)
# pred_holdout_old['pred_close']=np.exp(pred_holdout_old['pred_close'])
# pred_holdout_old

# sys.path.append("/kaggle/input/test-hack")
# from metrics import evaluate_all_metrics
# off_stats = evaluate_all_metrics(
#         y_true=holdout_act_y,
#         y_pred=pred_holdout_old,
#         x_test=pd.DataFrame({"window_id": holdout_set_X.index + 1,"time_step": 59,"close": holdout_set_X["timestamp_50_close"]}).reset_index(drop=True),
#         y_true_with_base=holdout_act_y.merge(pred_holdout_old,on=["window_id", "time_step"],how="inner",suffixes=("", "_pred")).rename(columns={"pred_close": "base_close"}),
#         horizon_step=0,
#     )
# display(pd.DataFrame([off_stats]).T.rename(columns={0: "value"}))